## CS441: Applied ML - PROJECT

# Emotion Detection Through Speech

*BY*

1.   **ABDULLAH SHAZAD**
2.   **AHMED MASOOD**
3.   **AHSAN KAMRAN**

In [1]:
import librosa
import soundfile as sf
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import gradio as gr
import joblib
from google.colab import files
import os
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.layers import Conv1D,MaxPooling1D,Flatten,Dense,Dropout,BatchNormalization

In [2]:
!pip install kaggle   #install kaggle

#uwrfkaggler/ravdess-emotional-speech-audio

In [3]:
files.upload()     #upload kaggle file

Saving kaggle.json to kaggle (1).json


{'kaggle (1).json': b'{"username":"i222179ahsankamran","key":"fb15eb6b7eb0a9b9f9ad63bccfac4435"}'}

In [4]:
!mkdir ~/.kaggle  #making folder of kaggle

In [5]:
!cp kaggle.json ~/.kaggle/   # copying the file

In [6]:
!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [7]:
!kaggle datasets download -d uwrfkaggler/ravdess-emotional-speech-audio

Dataset URL: https://www.kaggle.com/datasets/uwrfkaggler/ravdess-emotional-speech-audio
License(s): CC-BY-NC-SA-4.0
 77% 329M/429M [00:03<00:01, 53.1MB/s]
100% 429M/429M [00:04<00:00, 111MB/s] 


In [8]:
!unzip ravdess-emotional-speech-audio.zip -d ravdess_data

Archive:  ravdess-emotional-speech-audio.zip
  inflating: ravdess_data/Actor_01/03-01-01-01-01-01-01.wav  
  inflating: ravdess_data/Actor_01/03-01-01-01-01-02-01.wav  
  inflating: ravdess_data/Actor_01/03-01-01-01-02-01-01.wav  
  inflating: ravdess_data/Actor_01/03-01-01-01-02-02-01.wav  
  inflating: ravdess_data/Actor_01/03-01-02-01-01-01-01.wav  
  inflating: ravdess_data/Actor_01/03-01-02-01-01-02-01.wav  
  inflating: ravdess_data/Actor_01/03-01-02-01-02-01-01.wav  
  inflating: ravdess_data/Actor_01/03-01-02-01-02-02-01.wav  
  inflating: ravdess_data/Actor_01/03-01-02-02-01-01-01.wav  
  inflating: ravdess_data/Actor_01/03-01-02-02-01-02-01.wav  
  inflating: ravdess_data/Actor_01/03-01-02-02-02-01-01.wav  
  inflating: ravdess_data/Actor_01/03-01-02-02-02-02-01.wav  
  inflating: ravdess_data/Actor_01/03-01-03-01-01-01-01.wav  
  inflating: ravdess_data/Actor_01/03-01-03-01-01-02-01.wav  
  inflating: ravdess_data/Actor_01/03-01-03-01-02-01-01.wav  
  inflating: ravdess_data

In [9]:
!pip install librosa soundfile numpy pandas scikit-learn tensorflow gradio joblib

In [10]:
dataset='/content/ravdess_data/audio_speech_actors_01-24'
samples=22000  #22050
N_MFCC=40
max_pad_len=174
emotions={'01': 'neutral', '03': 'happy', '04': 'sad', '05': 'angry'}

In [11]:
def get_emotion_from_filename(fname):
    parts=fname.split('-')
    return parts[2] if len(parts) >= 3 else None

In [12]:
def extract_features(path):
    y,sr=librosa.load(path,sr=samples,mono=True)     #loading the audio file

    mfcc=librosa.feature.mfcc(y=y,sr=sr,n_mfcc=N_MFCC).T     #MFCC_features ectraction-> step time, shape

    if mfcc.shape[0] < max_pad_len:
        pad_amount = max_pad_len - mfcc.shape[0]
        mfcc = np.pad(mfcc, ((0, pad_amount), (0, 0)), mode="constant")     # # If MFCC is shorter than padding lenght -> pad it with zeros
    else:
        mfcc = mfcc[:max_pad_len]     ## If it's longer → cut it down to MAX_PAD_LEN
    return mfcc

In [13]:
files=[]
labels=[]

for root, _,filenames in os.walk(dataset):     # scan through every folder and file inside the dataset
    for fname in filenames:

        if fname.lower().endswith('.wav'):     # Only process .wav audio files

            emotion_code = get_emotion_from_filename(fname)      # Extract the emotion code from the file name

            if emotion_code in emotions:     # Keep the file only if its emotion is one we care about
                full_path = os.path.join(root, fname)
                files.append(full_path)     # save file path
                labels.append(emotions[emotion_code]) # save emotion label


In [14]:
X=[]
y=[]

for path,label in zip(files, labels):     # labelling of data

    features=extract_features(path)      # extracting MFCC features from the audio file
    if features is not None:      # adds if feature extraction worked
        X.append(features)
        y.append(label)

X=np.array(X)     # converts lists to NumPy arrays
y=np.array(y)
print("Dataset:", X.shape, len(y))


Dataset: (672, 174, 40) 672


In [15]:
# Turn text labels into numbers
encoder=LabelEncoder()
y_num=encoder.fit_transform(y)

y_onehot=to_categorical(y_num)     # Convert numbers into one-hot format

joblib.dump(encoder,"label_encoder.joblib")     # Save the encoder for later use


['label_encoder.joblib']

In [16]:
# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y_onehot, test_size=0.2, stratify=y, random_state=42)

In [17]:
#CNN MODEL
model = Sequential([
    Conv1D(64, 5, activation='relu', input_shape=(max_pad_len, N_MFCC)),
    BatchNormalization(),
    MaxPooling1D(2),
    Dropout(0.3),

    Conv1D(128, 5, activation='relu'),
    BatchNormalization(),
    MaxPooling1D(2),
    Dropout(0.3),

    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.4),
    Dense(y_onehot.shape[1], activation='softmax')
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [18]:
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

In [19]:
# Set up callbacks
checkpoint = ModelCheckpoint(
    "best_ravdess_model.h5",
    monitor="val_accuracy",
    save_best_only=True,
    verbose=1
)

early_stop = EarlyStopping(
    monitor="val_accuracy",
    patience=8,
    restore_best_weights=True
)

# Train the model
history = model.fit(
    X_train, y_train,
    validation_split=0.15,
    epochs=50,
    batch_size=32,
    callbacks=[checkpoint, early_stop]
)


Epoch 1/50
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 92ms/step - accuracy: 0.3748 - loss: 2.4014
Epoch 1: val_accuracy improved from -inf to 0.32099, saving model to best_ravdess_model.h5


15/15 ━━━━━━━━━━━━━━━━━━━━ 7s 167ms/step - accuracy: 0.3774 - loss: 2.3751 - val_accuracy: 0.3210 - val_loss: 6.5751
Epoch 2/50
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 194ms/step - accuracy: 0.5812 - loss: 1.1214
Epoch 2: val_accuracy improved from 0.32099 to 0.39506, saving model to best_ravdess_model.h5


15/15 ━━━━━━━━━━━━━━━━━━━━ 6s 244ms/step - accuracy: 0.5818 - loss: 1.1207 - val_accuracy: 0.3951 - val_loss: 3.9504
Epoch 3/50
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step - accuracy: 0.6004 - loss: 0.9129
Epoch 3: val_accuracy improved from 0.39506 to 0.44444, saving model to best_ravdess_model.h5


15/15 ━━━━━━━━━━━━━━━━━━━━ 2s 55ms/step - accuracy: 0.6032 - loss: 0.9101 - val_accuracy: 0.4444 - val_loss: 2.8097
Epoch 4/50
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step - accuracy: 0.7338 - loss: 0.7008
Epoch 4: val_accuracy did not improve from 0.44444
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 53ms/step - accuracy: 0.7316 - loss: 0.7039 - val_accuracy: 0.4444 - val_loss: 2.5884
Epoch 5/50
14/15 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step - accuracy: 0.7364 - loss: 0.6404
Epoch 5: val_accuracy improved from 0.44444 to 0.45679, saving model to best_ravdess_model.h5


15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 56ms/step - accuracy: 0.7343 - loss: 0.6472 - val_accuracy: 0.4568 - val_loss: 1.9181
Epoch 6/50
14/15 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step - accuracy: 0.7133 - loss: 0.6173
Epoch 6: val_accuracy did not improve from 0.45679
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 51ms/step - accuracy: 0.7173 - loss: 0.6112 - val_accuracy: 0.3951 - val_loss: 1.7716
Epoch 7/50
14/15 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step - accuracy: 0.7610 - loss: 0.5748
Epoch 7: val_accuracy improved from 0.45679 to 0.56790, saving model to best_ravdess_model.h5


15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 70ms/step - accuracy: 0.7629 - loss: 0.5710 - val_accuracy: 0.5679 - val_loss: 1.2842
Epoch 8/50
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step - accuracy: 0.8380 - loss: 0.4049
Epoch 8: val_accuracy did not improve from 0.56790
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 57ms/step - accuracy: 0.8373 - loss: 0.4056 - val_accuracy: 0.5062 - val_loss: 1.8012
Epoch 9/50
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step - accuracy: 0.8498 - loss: 0.4001
Epoch 9: val_accuracy improved from 0.56790 to 0.59259, saving model to best_ravdess_model.h5


15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 55ms/step - accuracy: 0.8491 - loss: 0.4011 - val_accuracy: 0.5926 - val_loss: 1.3375
Epoch 10/50
14/15 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step - accuracy: 0.8871 - loss: 0.2843
Epoch 10: val_accuracy improved from 0.59259 to 0.61728, saving model to best_ravdess_model.h5


15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 70ms/step - accuracy: 0.8842 - loss: 0.2905 - val_accuracy: 0.6173 - val_loss: 1.1326
Epoch 11/50
14/15 ━━━━━━━━━━━━━━━━━━━━ 0s 92ms/step - accuracy: 0.8993 - loss: 0.2908
Epoch 11: val_accuracy did not improve from 0.61728
15/15 ━━━━━━━━━━━━━━━━━━━━ 2s 103ms/step - accuracy: 0.8962 - loss: 0.2954 - val_accuracy: 0.5556 - val_loss: 1.1640
Epoch 12/50
14/15 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step - accuracy: 0.9043 - loss: 0.2347
Epoch 12: val_accuracy did not improve from 0.61728
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 98ms/step - accuracy: 0.9028 - loss: 0.2366 - val_accuracy: 0.5926 - val_loss: 1.4295
Epoch 13/50
14/15 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step - accuracy: 0.8554 - loss: 0.3675
Epoch 13: val_accuracy did not improve from 0.61728
15/15 ━━━━━━━━━━━━━━━━━━━━ 2s 78ms/step - accuracy: 0.8554 - loss: 0.3657 - val_accuracy: 0.6173 - val_loss: 1.2414
Epoch 14/50
14/15 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step - accuracy: 0.9084 - loss: 0.2223
Epoch 14: val_accuracy improved fro

15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 88ms/step - accuracy: 0.9059 - loss: 0.2251 - val_accuracy: 0.6543 - val_loss: 1.2547
Epoch 15/50
14/15 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - accuracy: 0.9419 - loss: 0.2057
Epoch 15: val_accuracy did not improve from 0.65432
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 52ms/step - accuracy: 0.9401 - loss: 0.2070 - val_accuracy: 0.6296 - val_loss: 1.3995
Epoch 16/50
14/15 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - accuracy: 0.9257 - loss: 0.1683
Epoch 16: val_accuracy did not improve from 0.65432
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 50ms/step - accuracy: 0.9251 - loss: 0.1700 - val_accuracy: 0.6049 - val_loss: 1.1051
Epoch 17/50
14/15 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step - accuracy: 0.9556 - loss: 0.1272
Epoch 17: val_accuracy did not improve from 0.65432
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 53ms/step - accuracy: 0.9554 - loss: 0.1281 - val_accuracy: 0.6296 - val_loss: 1.5079
Epoch 18/50
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - accuracy: 0.8978 - loss: 0.2638
Epoch 18: val_accuracy did not impro

In [20]:
loss,acc=model.evaluate(X_test,y_test)


5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.6515 - loss: 0.8750


In [21]:
model=tf.keras.models.load_model("best_ravdess_model.h5")
le=joblib.load("label_encoder.joblib")  # Load model + label encoder

# ---- Constants ----
SAMPLE_RATE=22050
N_MFCC=40
MAX_PAD_LEN=174

# ---- Feature Extraction ----
def extract_features(path):
    y,sr=librosa.load(path,sr=SAMPLE_RATE,mono=True)
    mfcc=librosa.feature.mfcc(y=y,sr=sr,n_mfcc=N_MFCC).T

    # Pad or trim to a fixed length
    if mfcc.shape[0]<MAX_PAD_LEN:
        pad_len=MAX_PAD_LEN-mfcc.shape[0]
        mfcc=np.pad(mfcc, ((0, pad_len), (0, 0)))
    else:
        mfcc=mfcc[:MAX_PAD_LEN]

    return np.expand_dims(mfcc,axis=0)  # shape for model

# ---- Prediction Function ----
def predict_emotion(audio):
    if audio is None:
        return "Please upload or record an audio file.", None

    # If microphone input → save temporary file
    if isinstance(audio,tuple):
        sr, data = audio
        sf.write("temp.wav", data, sr)
        audio = "temp.wav"

    features=extract_features(audio)
    preds=model.predict(features)[0]

    # Get predicted emotion
    idx=np.argmax(preds)
    emotion=le.inverse_transform([idx])[0]

    # Table of probabilities
    df = pd.DataFrame({
        "Emotion": le.classes_,
        "Probability (%)": [f"{p*100:.2f}%" for p in preds]
    })

    return f"Predicted Emotion:**{emotion.capitalize()}**",df

# ---- Gradio Interface ----
outlook=gr.Interface(
    fn=predict_emotion,
    inputs=gr.Audio(sources=["upload"], type="filepath", label="Upload Audio (.wav)"),
    outputs=[
        gr.Markdown(label="Prediction"),
        gr.Dataframe(label="Emotion Probabilities")
    ],
    title="Emotion Recognition Through Speech ",
    description="Upload an audio to predict emotions like neutral, happy, sad, angry."
)

outlook.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2100fefc0c59a8a130.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
